# Delta Lake: MERGE, Optimization & Debugging

This notebook demonstrates:
1. **MERGE INTO** — Upsert operations on Delta tables
2. **DESCRIBE HISTORY** — Full audit trail of every transaction (screenshot-ready)
3. **Liquid Clustering & Z-Ordering** — Data layout optimization
4. **Auto-compaction & Optimized Writes** — Automatic file management
5. **OPTIMIZE & VACUUM** — Table maintenance commands
6. **Interactive Debugger** — Step-through debugging in Databricks notebooks

> **Companion notebook**: *Delta Lake Time Travel & Schema Evolution* covers versioning and Bronze→Silver schema curation.

   
---
## 1.`DESCRIBE HISTORY`

Let's start by looking at the **real transaction history** of our Bronze accounts table. Notice the repeating **MERGE → OPTIMIZE** pattern — this is our production pipeline writing incremental upserts followed by automatic compaction.

In [0]:
%sql
DESCRIBE HISTORY bronze.cust_001.crm_accounts;

   
---
## 2. Delta Lake MERGE INTO

MERGE (upsert) is the backbone of incremental data pipelines. It lets us:
- **UPDATE** existing records when they match
- **INSERT** new records when they don't

Below we create a demo table in the Bronze layer, simulate incoming CRM updates, and merge them in.

In [0]:
%sql
    
-- Drop the old demo table if it exists, then create a fresh one with sample data
DROP TABLE IF EXISTS bronze.cust_001.accounts_merge_demo;

CREATE TABLE bronze.cust_001.accounts_merge_demo (
  id STRING,
  tenant BIGINT,
  source_system STRING,
  domain STRING,
  name STRING,
  status STRING,
  a_industry STRING,
  a_type STRING,
  a_annualrevenue STRING,
  updated_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql
    
-- Insert 6 sample accounts as our "existing" Bronze data
INSERT INTO bronze.cust_001.accounts_merge_demo VALUES
  ('ACC-001', 1, 'Salesforce', 'acme.com',        'Acme Corp',          'active',   'Technology',     'Customer', '5000000',  '2026-03-01T10:00:00'),
  ('ACC-002', 1, 'Salesforce', 'globex.com',       'Globex Industries',  'active',   'Manufacturing',  'Customer', '12000000', '2026-03-01T10:00:00'),
  ('ACC-003', 1, 'HubSpot',   'initech.io',       'Initech Solutions',  'active',   'Consulting',     'Prospect', '800000',   '2026-03-01T10:00:00'),
  ('ACC-004', 1, 'Salesforce', 'umbrella-corp.net','Umbrella Corp',      'active',   'Pharmaceuticals','Customer', '45000000', '2026-03-01T10:00:00'),
  ('ACC-005', 1, 'HubSpot',   'stark.io',         'Stark Enterprises',  'active',   'Technology',     'Prospect', '90000000', '2026-03-01T10:00:00'),
  ('ACC-006', 1, 'Salesforce', 'wayne.com',        'Wayne Industries',   'inactive', 'Conglomerate',   'Customer', '70000000', '2026-03-01T10:00:00');

SELECT * FROM bronze.cust_001.accounts_merge_demo ORDER BY id;

In [0]:
%sql
-- Simulate incoming CRM batch: 2 updates + 2 brand new accounts
CREATE OR REPLACE TEMP VIEW staging_accounts AS
SELECT * FROM VALUES
  -- UPDATE: Acme Corp got acquired, revenue changed
  ('ACC-001', 1, 'Salesforce', 'acme.com',        'Acme Corp (Acquired)',   'active',   'Technology',     'Customer',  '15000000', TIMESTAMP '2026-04-01 08:00:00'),
  -- UPDATE: Wayne Industries reactivated
  ('ACC-006', 1, 'Salesforce', 'wayne.com',        'Wayne Industries',       'active',   'Conglomerate',   'Customer',  '85000000', TIMESTAMP '2026-04-01 08:00:00'),
  -- INSERT: Brand new prospect
  ('ACC-007', 1, 'HubSpot',   'oscorp.com',       'Oscorp Technologies',    'active',   'Biotechnology',  'Prospect',  '3000000',  TIMESTAMP '2026-04-01 08:00:00'),
  -- INSERT: Brand new customer
  ('ACC-008', 1, 'Salesforce', 'lexcorp.com',      'LexCorp',                'active',   'Conglomerate',   'Customer',  '60000000', TIMESTAMP '2026-04-01 08:00:00')
AS t(id, tenant, source_system, domain, name, status, a_industry, a_type, a_annualrevenue, updated_at);

-- Preview what's coming in
SELECT id, name, status, a_annualrevenue, updated_at, 
       CASE 
         WHEN id IN ('ACC-001','ACC-006') THEN '⬆️ UPDATE' 
         ELSE '➕ INSERT' 
       END AS merge_action
FROM staging_accounts
ORDER BY id;

In [0]:
%sql
    
MERGE INTO bronze.cust_001.accounts_merge_demo AS target
USING staging_accounts AS source
  ON target.id = source.id

WHEN MATCHED THEN UPDATE SET
  target.name            = source.name,
  target.status          = source.status,
  target.a_industry      = source.a_industry,
  target.a_type          = source.a_type,
  target.a_annualrevenue = source.a_annualrevenue,
  target.updated_at      = source.updated_at

WHEN NOT MATCHED THEN INSERT *;

---
## 3. Delta Table Optimization

### Liquid Clustering vs Z-Ordering
| Feature | Liquid Clustering | Z-Ordering |
| --- | --- | --- |
| **Syntax** | `CLUSTER BY (cols)` | `OPTIMIZE ... ZORDER BY (cols)` |
| **When applied** | Incrementally on writes | Manual via OPTIMIZE |
| **Re-clustering** | Automatic | Must re-run OPTIMIZE |
| **Recommended** | New tables (DBR 13.3+) | Legacy or existing tables |

### Auto-compaction & Optimized Writes
| Feature | What It Does |
| --- | --- |
| **Auto-compaction** | Automatically merges small files after writes |
| **Optimized writes** | Coalesces small partitions during writes to reduce file count |

In [0]:
%sql
    
-- Enable Liquid Clustering on our demo table
-- This tells Delta to automatically cluster data by these columns during writes
ALTER TABLE bronze.cust_001.accounts_merge_demo
CLUSTER BY (source_system, a_industry);

In [0]:
%sql
    
-- Enable automatic file management on the demo table
ALTER TABLE bronze.cust_001.accounts_merge_demo SET TBLPROPERTIES (
  'delta.autoOptimize.autoCompact' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true'
);

-- Verify the properties
SHOW TBLPROPERTIES bronze.cust_001.accounts_merge_demo;